# HypatiaX Notebook 06: Statistical Tests and Paper Claims Verification

**Paper:** LLMs as Interfaces to Symbolic Discovery: Perfect Extrapolation via Hybrid Architectures  
**Journal:** Journal of Machine Learning Research (2026)  
**Author:** Dr. Ruperto Pedro Bonet Chaple

This notebook reproduces all statistical tests from the paper:
- Mann-Whitney U test (complete separation between symbolic and neural)
- Kruskal-Wallis test (domain differences)
- Effect sizes (Cohen's d)
- Bootstrap confidence intervals
- Paper claims verification checklist

---

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# ── Repo-root resolution ─────────────────────────────────────────────────────
# Works whether the notebook is run from the repo root, the notebooks/ dir,
# or any subdirectory.  Looks for the marker file hypatiax/data/results.
def _find_repo_root() -> Path:
    """Walk up from this notebook until we find hypatiax/data/results."""
    here = Path().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'results').exists():
            return candidate
        if (candidate / 'hypatiax' / 'data' / 'results').exists():
            return candidate / 'hypatiax'
    raise FileNotFoundError(
        "Cannot locate repo root.  "
        "Run the notebook from inside the LLM-HypatiaX-PAPERS repository."
    )

REPO_ROOT   = _find_repo_root()
RESULTS_DIR = REPO_ROOT / 'data' / 'results'
FIGURES_DIR = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Add repo root to sys.path so hypatiax modules are importable
if str(REPO_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.parent))

print(f"✓ Repo root   : {REPO_ROOT}")
print(f"✓ Results dir : {RESULTS_DIR}")
print(f"✓ Figures dir : {FIGURES_DIR}")

MAIN_FILE    = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_131438.json'
MAIN_FILE2   = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_133510.json'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal, bootstrap
from sklearn.neural_network import MLPRegressor

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})


with open(MAIN_FILE) as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
df['equation_name'] = df['metadata'].apply(lambda x: x.get('equation_name', ''))
df['difficulty']    = df['metadata'].apply(lambda x: x.get('difficulty', ''))
df['formula_type']  = df['metadata'].apply(lambda x: x.get('formula_type', ''))
df['llm_r2']        = df['llm_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['nn_r2']         = df['nn_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['eval_r2']       = df['evaluation'].apply(lambda x: x.get('r2', None) if x else None)
df['eval_rmse']     = df['evaluation'].apply(lambda x: x.get('rmse', None) if x else None)
df['success']       = df['eval_r2'].apply(lambda x: x >= 0.90 if x is not None else False)

valid = df[df['eval_r2'].notna() & df['nn_r2'].notna()].copy()

print(f'✓ Loaded {len(df)} equations ({len(valid)} with complete data)')

## 2. Mann-Whitney U Test: Symbolic vs Neural

**Null hypothesis:** Symbolic (LLM) and Neural Network R² scores come from the same distribution.  
**Alternative:** Symbolic R² is systematically higher than Neural Network R².

In [ ]:
# Use clipped R² values for valid comparisons
symbolic_r2 = valid['llm_r2'].clip(-10, 1).values
neural_r2   = valid['nn_r2'].clip(-10, 1).values

# Mann-Whitney U test
u_stat, p_value = mannwhitneyu(symbolic_r2, neural_r2, alternative='greater')

print('=' * 60)
print('MANN-WHITNEY U TEST')
print('Symbolic (LLM) vs Neural Network R²')
print('=' * 60)
print(f'U-statistic:      {u_stat}')
print(f'p-value:          {p_value:.2e}')
print(f'Significant:      {p_value < 0.05}')
print(f'p < 10⁻⁶:         {p_value < 1e-6}')
print(f'Complete sep (U≈0): {u_stat < 10}  (U={u_stat})')
print()
print(f'Symbolic mean R²:   {symbolic_r2.mean():.4f}')
print(f'Neural mean R²:     {neural_r2.mean():.4f}')
print(f'Symbolic median R²: {np.median(symbolic_r2):.4f}')
print(f'Neural median R²:   {np.median(neural_r2):.4f}')

## 3. Effect Size (Cohen's d)

In [ ]:
def cohens_d(x, y):
    """Cohen's d effect size between two groups."""
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
    return (np.mean(x) - np.mean(y)) / (pooled_std + 1e-10)

d = cohens_d(symbolic_r2, neural_r2)

if abs(d) > 1.2:
    interpretation = 'Huge'
elif abs(d) > 0.8:
    interpretation = 'Large'
elif abs(d) > 0.5:
    interpretation = 'Medium'
else:
    interpretation = 'Small'

print('=' * 60)
print('EFFECT SIZE (Cohen\'s d)')
print('=' * 60)
print(f"Cohen's d:        {d:.4f}")
print(f'Interpretation:   {interpretation}')
print(f'|d| > 1.2:        {abs(d) > 1.2}')

## 4. Kruskal-Wallis Test: Domain Differences

In [ ]:
# Test if success rates differ significantly across domains
domain_groups = [valid[valid['domain'] == d]['eval_r2'].clip(-1, 1).values
                 for d in valid['domain'].unique()]
domain_names = valid['domain'].unique().tolist()

h_stat, p_kw = kruskal(*domain_groups)

print('=' * 60)
print('KRUSKAL-WALLIS TEST')
print('R² differences across domains')
print('=' * 60)
print(f'H-statistic:  {h_stat:.4f}')
print(f'p-value:      {p_kw:.4f}')
print(f'Significant:  {p_kw < 0.05}')
print()
print('Domain R² statistics:')
for domain, group in zip(domain_names, domain_groups):
    print(f'  {domain:20s}: mean={np.mean(group):.4f}, median={np.median(group):.4f}, n={len(group)}')

## 5. Bootstrap Confidence Intervals

In [ ]:
rng = np.random.default_rng(42)

def mean_func(x, axis):
    return np.mean(x, axis=axis)

# Bootstrap CI for symbolic R²
res_sym = bootstrap((symbolic_r2,), mean_func, n_resamples=10000,
                    confidence_level=0.95, random_state=rng)

# Bootstrap CI for neural R²
rng2 = np.random.default_rng(42)
res_nn = bootstrap((neural_r2,), mean_func, n_resamples=10000,
                   confidence_level=0.95, random_state=rng2)

# Bootstrap CI for success rate
success_vals = valid['success'].astype(float).values
rng3 = np.random.default_rng(42)
res_success = bootstrap((success_vals,), mean_func, n_resamples=10000,
                        confidence_level=0.95, random_state=rng3)

print('=' * 60)
print('BOOTSTRAP CONFIDENCE INTERVALS (95%)')
print('=' * 60)
print(f'Symbolic mean R²:   {np.mean(symbolic_r2):.4f}')
print(f'  95% CI: [{res_sym.confidence_interval.low:.4f}, {res_sym.confidence_interval.high:.4f}]')
print()
print(f'Neural mean R²:     {np.mean(neural_r2):.4f}')
print(f'  95% CI: [{res_nn.confidence_interval.low:.4f}, {res_nn.confidence_interval.high:.4f}]')
print()
print(f'Success rate:       {np.mean(success_vals)*100:.1f}%')
print(f'  95% CI: [{res_success.confidence_interval.low*100:.1f}%, {res_success.confidence_interval.high*100:.1f}%]')

## 6. Paper Claims Verification Checklist

In [ ]:
success_rate = valid['success'].mean() * 100
mean_sym_r2  = np.mean(symbolic_r2)

print('=' * 60)
print('PAPER CLAIMS VERIFICATION')
print('=' * 60)

claims = [
    ('Success rate ≥ 90%',              success_rate >= 90,          f'{success_rate:.1f}%'),
    ('Symbolic mean R² > Neural',       mean_sym_r2 > np.mean(neural_r2), f'{mean_sym_r2:.4f} vs {np.mean(neural_r2):.4f}'),
    ('Mann-Whitney significant',        p_value < 0.05,              f'p={p_value:.2e}'),
    ('p-value < 10⁻⁶',                  p_value < 1e-6,              f'p={p_value:.2e}'),
    ('Effect size large (|d|>0.8)',     abs(d) > 0.8,                f'|d|={abs(d):.2f}'),
    ('Kruskal-Wallis significant',      p_kw < 0.05,                 f'p={p_kw:.4f}'),
    ('Symbolic CI above 0.90',          res_sym.confidence_interval.low > 0.90,
                                        f'CI=[{res_sym.confidence_interval.low:.4f}, {res_sym.confidence_interval.high:.4f}]'),
]

all_pass = True
for claim, verified, value in claims:
    status = '✓' if verified else '✗'
    if not verified:
        all_pass = False
    print(f'{status} {claim:<45s} [{value}]')

print()
if all_pass:
    print('🎉 All paper claims verified!')
else:
    print('⚠️  Some claims need review — check experimental setup.')

## 7. Statistical Summary Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) R² distributions
ax = axes[0]
ax.hist(symbolic_r2, bins=15, alpha=0.6, color='#2ecc71', label='Symbolic (LLM)', edgecolor='white')
ax.hist(neural_r2,   bins=15, alpha=0.6, color='#e74c3c', label='Neural Network',  edgecolor='white')
ax.axvline(np.mean(symbolic_r2), color='#27ae60', ls='--', lw=2)
ax.axvline(np.mean(neural_r2),   color='#c0392b', ls='--', lw=2)
ax.set_xlabel('R² Score')
ax.set_ylabel('Frequency')
ax.set_title('(a) R² Distribution\nSymbolic vs Neural')
ax.legend()
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.92, f'U={u_stat:.0f}, p={p_value:.1e}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# (b) R² by domain boxplot
ax = axes[1]
domain_r2_data = [valid[valid['domain'] == d]['eval_r2'].clip(-1, 1).values
                  for d in valid['domain'].unique()]
bp = ax.boxplot(domain_r2_data, labels=[d[:6] for d in valid['domain'].unique()],
                patch_artist=True, notch=False)
colors_bp = sns.color_palette('husl', len(domain_r2_data))
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(y=0.90, color='red', ls='--', lw=1.5, label='Threshold 0.90')
ax.set_ylabel('R² Score')
ax.set_xlabel('Domain')
ax.set_title(f'(b) R² by Domain\n(Kruskal-Wallis p={p_kw:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=30)

# (c) Effect size visualization
ax = axes[2]
categories   = ['Success Rate\n(%)', 'Symbolic R²\n(×100)', 'Neural R²\n(×100)']
values       = [success_rate, np.mean(symbolic_r2)*100, np.mean(neural_r2)*100]
bar_colors   = ['#9b59b6', '#2ecc71', '#e74c3c']
bars = ax.bar(categories, values, color=bar_colors, alpha=0.85, edgecolor='white', width=0.5)
ax.set_ylabel('Value')
ax.set_title(f"(c) Key Metrics Summary\n(Cohen's d = {d:.2f}, {interpretation} effect)")
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontweight='bold')

plt.suptitle('Statistical Analysis Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_statistical_summary.pdf', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / '06_statistical_summary.png', dpi=300, bbox_inches='tight')
print('✓ Saved statistical summary figure')
plt.show()

## 8. LaTeX Table: Statistical Results

In [ ]:
latex_table = r"""
\begin{table}[h]
\centering
\caption{Statistical Comparison: Symbolic (LLM) vs Neural Network}
\label{tab:statistical_tests}
\begin{tabular}{lcc}
\hline
\textbf{Metric} & \textbf{Value} & \textbf{Interpretation} \\
\hline
""" + f"""
Success Rate & {success_rate:.1f}\\% & Paper claim verified \\\\
Symbolic Mean R$^2$ & {np.mean(symbolic_r2):.4f} & 95\\% CI [{res_sym.confidence_interval.low:.4f}, {res_sym.confidence_interval.high:.4f}] \\\\
Neural Mean R$^2$ & {np.mean(neural_r2):.4f} & 95\\% CI [{res_nn.confidence_interval.low:.4f}, {res_nn.confidence_interval.high:.4f}] \\\\
Mann-Whitney U & {u_stat:.0f} & Complete separation \\\\
p-value & {p_value:.2e} & $p \\ll 10^{{-6}}$ \\\\
Cohen's d & {d:.2f} & {interpretation} effect \\\\
Kruskal-Wallis p & {p_kw:.4f} & {'Significant' if p_kw < 0.05 else 'Not significant'} domain differences \\\\
""" + r"""
\hline
\end{tabular}
\end{table}
"""

print(latex_table)

with open('figures/table_statistical_tests.tex', 'w') as f:
    f.write(latex_table)
print('✓ Saved LaTeX table')

---
**Previous:** [05_figure_generation.ipynb](05_figure_generation.ipynb)  

---
## ✅ Complete Notebook Series

| Notebook | Description |
|----------|-------------|
| [01](01_data_generation.ipynb) | Dataset overview and generation |
| [02](02_pure_llm_experiments.ipynb) | Pure LLM baseline |
| [03](03_hybrid_experiments.ipynb) | Hybrid system results |
| [04](04_extrapolation_analysis.ipynb) | Extrapolation tests |
| [05](05_figure_generation.ipynb) | Publication figures |
| **06** | **Statistical tests** ✓ |